In [9]:
import pandas as pd
import numpy as np
import json

# --- INGESTION ---
try:
    with open('../data/raw_credit_applications.json', 'r') as f:
        df = pd.json_normalize(json.load(f))
    print(f"[SYSTEM] Logged: {len(df)} records ingested.")
except Exception as e:
    print(f"[ERROR] Ingestion failed: {e}")

# --- DATA QUALITY REPORT ---
def generate_dq_report(df):
    print("\n" + "="*60)
    print(" NOVACRED DATA QUALITY REPORT v1.0")
    print("="*60)
    
    # 1. Duplicate Records
    dupes = df.duplicated(subset=['_id']).sum()
    print(f"I.   UNIQUENESS: {dupes} duplicate(s) identified ({(dupes/len(df)*100):.2f}%)")

    # 2. Inconsistent Data Types
    print("\nII.  CONSISTENCY (DATA TYPES):")
    if 'financials.annual_income' in df.columns:
        types = df['financials.annual_income'].apply(type).value_counts()
        for dtype, count in types.items():
            print(f"     - financials.annual_income | {dtype}: {count}")

    # 3. Missing or Incomplete Records
    print("\nIII. COMPLETENESS (TOP MISSING FIELDS):")
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df)) * 100
    missing_report = pd.DataFrame({'counts': null_counts, 'pct': null_pct})
    top_missing = missing_report[missing_report['counts'] > 0].sort_values(by='pct', ascending=False)
    for field, row in top_missing.iterrows():
        print(f"     - {field:30} | {int(row['counts']):3} missing | {row['pct']:5.1f}%")

    # 4. Inconsistent Coding/Formatting (Categorical)
    print("\nIV.  CONSISTENCY (CATEGORICAL CODING):")
    if 'applicant_info.gender' in df.columns:
        unique_vals = [str(x) for x in df['applicant_info.gender'].unique()]
        print(f"     - applicant_info.gender | Observed: {', '.join(unique_vals)}")

    # 5. Invalid or Impossible Values
    print("\nV.   VALIDITY (DOMAIN CONSTRAINTS):")
    if 'financials.credit_history_months' in df.columns:
        invalid = (df['financials.credit_history_months'] < 0).sum()
        print(f"     - financials.credit_history_months | Negatives detected: {invalid}")

    # 6. Inconsistent Date Formats
    print("\nVI.  ACCURACY (TEMPORAL FORMATS):")
    if 'applicant_info.date_of_birth' in df.columns:
        samples = df['applicant_info.date_of_birth'].head(8).to_list()
        print(f"     - applicant_info.date_of_birth | Samples: {samples}")
    
    print("="*60)

generate_dq_report(df)

[SYSTEM] Logged: 502 records ingested.

 NOVACRED DATA QUALITY REPORT v1.0
I.   UNIQUENESS: 2 duplicate(s) identified (0.40%)

II.  CONSISTENCY (DATA TYPES):
     - financials.annual_income | <class 'int'>: 488
     - financials.annual_income | <class 'str'>: 8
     - financials.annual_income | <class 'float'>: 6

III. COMPLETENESS (TOP MISSING FIELDS):
     - notes                          | 500 missing |  99.6%
     - financials.annual_salary       | 497 missing |  99.0%
     - loan_purpose                   | 452 missing |  90.0%
     - processing_timestamp           | 440 missing |  87.6%
     - decision.rejection_reason      | 292 missing |  58.2%
     - decision.interest_rate         | 210 missing |  41.8%
     - decision.approved_amount       | 210 missing |  41.8%
     - applicant_info.ssn             |   5 missing |   1.0%
     - applicant_info.ip_address      |   5 missing |   1.0%
     - financials.annual_income       |   5 missing |   1.0%
     - applicant_info.gender      